# Check Data Downloaded/Uploaded from GDrive

In [1]:
! ls /home/shadeform/Data

expression.csv	   labels_MOT1_B.csv	    labels_SUB2.csv
labels_DOM.csv	   labels_MOT2_A.csv	    protein_embeddings.npz
labels_MOT1_A.csv  labels_NULL_INDEP_A.csv


# Imports

In [3]:
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

import prolm
from prolm.data import ProteinClassificationDataset, load_prolm_data
from prolm.model import (
    ProLMBackbone,
    ProLMClassifier,
    freeze_backbone,
    infer_checkpoint_shape,
    load_matching_state_dict,
    strip_module_prefix,
)

# Function definitions from finetune.py

In [84]:

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True


def build_model(model_params, data, device: torch.device) -> ProLMClassifier:
    feature_dim = int(data.features_tensor.shape[1])
    hidden_size = model_params.hidden_size
    num_layers = model_params.num_layers
    chunk_size = model_params.chunk_size
    state_dict = None
    if model_params.pretrained_checkpoint:
        state_dict = torch.load(model_params.pretrained_checkpoint, map_location="cpu")
        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        state_dict = strip_module_prefix(state_dict)
        shape = infer_checkpoint_shape(state_dict)
        feature_dim = shape.feature_dim
        hidden_size = shape.hidden_size
        num_layers = shape.num_layers
        chunk_size = shape.chunk_size
        if data.features_tensor.shape[1] != feature_dim:
            raise ValueError(
                "Embedding dimension does not match checkpoint: "
                f"data has {data.features_tensor.shape[1]}, checkpoint expects {feature_dim}."
            )

    backbone = ProLMBackbone(
        feature_dim=feature_dim,
        hidden_size=hidden_size,
        num_layers=num_layers,
        num_proteins=len(data.protein_names),
        features_tensor=data.features_tensor,
        bias_matrix=data.bias_matrix,
        chunk_size=chunk_size,
    )
    if state_dict is not None:
        load_result, skipped = load_matching_state_dict(
            backbone,
            state_dict,
            skip_keys={"features_tensor", "bias_matrix_full"},
        )
        if skipped:
            print(f"Skipped {len(skipped)} checkpoint tensors with incompatible names or shapes.")
        if load_result.missing_keys:
            print(f"Missing after partial checkpoint load: {load_result.missing_keys[:5]}")
        if load_result.unexpected_keys:
            print(f"Unexpected after partial checkpoint load: {load_result.unexpected_keys[:5]}")
    
    freeze_backbone(backbone, model_params.unfreeze_last_n_layers)
    model = ProLMClassifier(backbone, num_classes=model_params.output_size).to(device)
    
    return model


def evaluate(model: ProLMClassifier, loader: DataLoader, device: torch.device) -> dict[str, float]:
    model.eval()
    y_true = []
    y_prob = []
    with torch.no_grad():
        for x, y, _ in loader:
            logits = model(x.to(device))
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            y_prob.extend(probs.tolist())
            y_true.extend(y.numpy().tolist())
    y_pred = [1 if p >= 0.5 else 0 for p in y_prob]
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auprc": float(average_precision_score(y_true, y_prob)),
    }
    metrics["auroc"] = float(roc_auc_score(y_true, y_prob)) if len(set(y_true)) > 1 else float("nan")
    return metrics


def save_patient_embeddings(
    model: ProLMClassifier, data, device: torch.device, path: Path, batch_size: int = 16
) -> pd.DataFrame:
    """Save one ProLM backbone vector per patient, with its eid.

    The backbone splits the proteins into chunks, runs each chunk with its own [CLS] token and averages the
    chunk [CLS] vectors, so the result has shape (n_patients, hidden_size). Evaluation mode switches dropout
    off (the vectors are reproducible), no_grad avoids building a gradient graph, and patients are processed
    in batches on the model's device.
    """
    was_training = model.training
    model.eval()
    expressions = torch.tensor(data.expression.values, dtype=torch.float32)
    parts = []
    with torch.no_grad():
        for start in range(0, len(expressions), batch_size):
            parts.append(model.backbone(expressions[start:start + batch_size].to(device)).cpu())
    values = torch.cat(parts).numpy()
    table = pd.DataFrame(values, index=data.expression.index, columns=[f"e{j}" for j in range(values.shape[1])])
    table.index.name = "eid"
    table.to_csv(path)
    model.train(was_training)
    return table



# Step through finetune.py main()

## Device Setup

In [14]:
set_seed(314159)
output_dir = Path('/home/shadeform/Output/')
output_dir.mkdir(parents=True, exist_ok=True)
device = 'cuda'
if device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but no GPU is available. Use --device cpu to run on CPU.")
device = torch.device(device)
print(device)

cuda


## Make multiclass labels

In [67]:
label_features = [
    'DOM',
    'MOT1_A',
    'MOT1_B',
    'MOT2_A',
    'NULL_INDEP_A',
    'SUB2'
]

new_features = pd.concat([
    pd.read_csv(f'/home/shadeform/Data/labels_{f}.csv', index_col='eid').rename(columns={'label': f})
    for f in label_features
], axis=1)

new_features.to_csv('/home/shadeform/Data/labels_multiclass.csv')
new_features.head()

,DOM,MOT1_A,MOT1_B,MOT2_A,NULL_INDEP_A,SUB2
eid,,,,,,
S001,1,0,0,0,0,0
S002,0,0,0,1,0,0
S003,1,0,0,0,0,0
S004,1,0,0,1,0,0
S005,1,0,0,1,1,1


## Data Setup

In [81]:
expression_csv = '/home/shadeform/Data/expression.csv'
labels_csv = '/home/shadeform/Data/labels_multiclass.csv'
protein_embeddings = '/home/shadeform/Data/protein_embeddings.npz'
correlation_csv = None
eid_col = 'eid'
label_col = 'DOM'
batch_size = 128

In [82]:
data = load_prolm_data(
    expression_csv,
    labels_csv,
    protein_embeddings,
    correlation_csv,
    eid_col=eid_col,
    label_col=label_col,
)

label_names = label_features

dataset = ProteinClassificationDataset(data.expression, data.labels)
labels = pd.read_csv(labels_csv, index_col='eid').to_numpy()

indices = np.arange(len(dataset))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.5,
    random_state=314159
)

train_loader = DataLoader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(Subset(dataset, test_idx), batch_size=batch_size, shuffle=False)

In [83]:
print(len(train_idx), len(test_idx))
print(len(train_loader), len(test_loader))

100 100
1 1


## Model Setup

In [96]:
model_args = {
    'hidden_size': 32,
    'num_layers': 1,
    'chunk_size': 16,
    'pretrained_checkpoint': None,
    'unfreeze_last_n_layers': 0,
    'output_size': 6
}

from types import SimpleNamespace
model_args = SimpleNamespace(**model_args)

In [97]:
model = build_model(model_args, data, device)

if not model_args.pretrained_checkpoint:
    # Without a checkpoint there is nothing to keep frozen: build_model always calls freeze_backbone, which
    # would leave the encoder, the feature/expression projections and the [CLS] token at their random values.
    for parameter in model.parameters():
        parameter.requires_grad = True

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_trainable:,} of {sum(p.numel() for p in model.parameters()):,}")

Trainable parameters: 17,319 of 17,319


In [98]:
from torchinfo import summary

summary(
    model,
    input_size=(batch_size, 7000),  # batch_size, num_features
)

Layer (type:depth-idx)                                                      Output Shape              Param #
ProLMClassifier                                                             [128, 6]                  --
├─ProLMBackbone: 1-1                                                        [128, 32]                 33
│    └─Linear: 2-1                                                          [7000, 32]                2,080
│    └─Linear: 2-2                                                          [128, 7000, 32]           64
│    └─CorrelationBiasedBertModel: 2-3                                      [128, 17, 32]             2,240
│    │    └─CorrelationBiasedBertEncoder: 3-1                               [128, 17, 32]             12,704
│    └─CorrelationBiasedBertModel: 2-4                                      [128, 17, 32]             (recursive)
│    │    └─CorrelationBiasedBertEncoder: 3-2                               [128, 17, 32]             (recursive)
│    └─CorrelationBias

## Training Loop

In [107]:
# Training params

num_epochs = 20

In [108]:
history = []
best_auroc = -1.0

optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=1E-3)
criterion = torch.nn.BCEWithLogitsLoss()

for epoch in range(1, num_epochs + 1):
    model.train()
    losses = []
    for x, y, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}", leave=False):
        x = x.to(device)
        y = torch.tensor(labels[train_idx].astype(float)).to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))

    train_metrics = evaluate(model, train_loader, device)
    test_metrics = evaluate(model, test_loader, device)
    metrics = {f'train_{k}': v for k, v in train_metrics.items()} | {f'test_{k}': v for k, v in test_metrics.items()}
    metrics["epoch"] = epoch
    metrics["train_loss"] = float(np.mean(losses))
    metrics['test_loss'] = float(criterion(logits, torch.tensor(labels[test_idx].astype(float)).to(device)))
    history.append(metrics)
    print(json.dumps(metrics, indent=2))
    score = metrics["test_auroc"] if not np.isnan(metrics["test_auroc"]) else metrics["test_accuracy"]
    if score > best_auroc:
        best_auroc = score
        torch.save(model.state_dict(), output_dir / "best_classifier.pt")
        # separate saving of the backbone (the part that makes the patient embeddings)
        torch.save(model.backbone.state_dict(), output_dir / "backbone.pt")


Epoch 1/20:   0%|          | 0/1 [00:00<?, ?it/s]/home/shadeform/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/home/shadeform/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


{
  "train_accuracy": 0.4,
  "train_f1": 0.0,
  "train_auprc": 0.6215727825927366,
  "train_auroc": 0.5045833333333334,
  "test_accuracy": 0.34,
  "test_f1": 0.0,
  "test_auprc": 0.7049743276026679,
  "test_auroc": 0.5467914438502673,
  "epoch": 1,
  "train_loss": 0.7148798400970797,
  "test_loss": 0.7083658825419844
}


Epoch 2/20:   0%|          | 0/1 [00:00<?, ?it/s]/home/shadeform/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/home/shadeform/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


{
  "train_accuracy": 0.4,
  "train_f1": 0.0,
  "train_auprc": 0.6220839886708615,
  "train_auroc": 0.5091666666666668,
  "test_accuracy": 0.34,
  "test_f1": 0.0,
  "test_auprc": 0.7081045196954837,
  "test_auroc": 0.5443404634581106,
  "epoch": 2,
  "train_loss": 0.7048432261000077,
  "test_loss": 0.7012525642228624
}


Epoch 3/20:   0%|          | 0/1 [00:00<?, ?it/s]/home/shadeform/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/home/shadeform/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


{
  "train_accuracy": 0.4,
  "train_f1": 0.0,
  "train_auprc": 0.6325678410374641,
  "train_auroc": 0.510625,
  "test_accuracy": 0.34,
  "test_f1": 0.0,
  "test_auprc": 0.7175436669522093,
  "test_auroc": 0.5487967914438503,
  "epoch": 3,
  "train_loss": 0.6802574165165425,
  "test_loss": 0.6755288675862055
}


... (30 outputs left out)

Epoch 19/20:   0%|          | 0/1 [00:00<?, ?it/s]/home/shadeform/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/home/shadeform/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


{
  "train_accuracy": 0.4,
  "train_f1": 0.0,
  "train_auprc": 0.6080544669623983,
  "train_auroc": 0.4945833333333333,
  "test_accuracy": 0.34,
  "test_f1": 0.0,
  "test_auprc": 0.6745202627039455,
  "test_auroc": 0.4881907308377897,
  "epoch": 19,
  "train_loss": 0.5734759620080392,
  "test_loss": 0.5476982034246127
}


Epoch 20/20:   0%|          | 0/1 [00:00<?, ?it/s]/home/shadeform/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/home/shadeform/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


{
  "train_accuracy": 0.4,
  "train_f1": 0.0,
  "train_auprc": 0.6078264634425444,
  "train_auroc": 0.4947916666666666,
  "test_accuracy": 0.34,
  "test_f1": 0.0,
  "test_auprc": 0.6752683744018209,
  "test_auroc": 0.4870766488413547,
  "epoch": 20,
  "train_loss": 0.5808443614964685,
  "test_loss": 0.5634852198014657
}


In [109]:
# Patient embeddings from the selected model (the one saved as best_classifier.pt)
model.load_state_dict(torch.load(output_dir / "best_classifier.pt", map_location=device))
embeddings = save_patient_embeddings(model, data, device, output_dir / "patient_embeddings.csv")
print(f"Saved patient embeddings: {embeddings.shape[0]} patients x {embeddings.shape[1]} dimensions")

(output_dir / "metrics.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
(output_dir / "protein_order.txt").write_text("\n".join(data.protein_names) + "\n", encoding="utf-8")
print(f"Saved outputs to {output_dir}")

/home/shadeform/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Saved patient embeddings: 200 patients x 32 dimensions
Saved outputs to /home/shadeform/Output
